# Interview Transcriber — launcher

Use a T4 GPU runtime and the same Google account whose Drive contains the recordings. Store the Hugging Face read-only token once as HF_TOKEN in Colab Secrets. Run the single START INTERVIEW TRANSCRIBER cell below. It mounts Drive, updates the repo, installs dependencies, and launches the temporary Gradio UI.

In [ ]:
# @title ▶ START INTERVIEW TRANSCRIBER
import os
import secrets
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

REPO_URL = "https://github.com/playply/transcriber.git"
REPO_DIR = Path("/content/transcriber")
DRIVE_MOUNT = Path("/content/drive")

# 1) Require a GPU before doing any heavy setup.
nvidia_smi = shutil.which("nvidia-smi")
if not nvidia_smi:
    raise RuntimeError("NVIDIA GPU is not attached to this Colab runtime. Select Runtime → Change runtime type → T4 GPU, reconnect the runtime, then run START again.")
gpu_check = subprocess.run(
    [nvidia_smi],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if gpu_check.returncode != 0:
    raise RuntimeError("NVIDIA GPU check failed. In Colab select Runtime → Change runtime type → T4 GPU, reconnect the runtime, then run START again.")
print("✓ GPU available")

# 2) Mount the same Google account whose Drive contains the recordings.
drive.mount(str(DRIVE_MOUNT))
print("✓ Google Drive mounted")

# 3) Read the Hugging Face read-only token from Colab Secrets.
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "HF_TOKEN is not available. Add a read-only HF_TOKEN in Colab Secrets and enable notebook access."
    ) from exc
if not hf_token:
    raise RuntimeError("HF_TOKEN is empty in Colab Secrets.")
print("✓ HF_TOKEN loaded from Colab Secrets")

# 4) Get the latest application code.
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "HEAD"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
print("✓ Repository ready")

# 5) Install a dependency set compatible with the tested WhisperX baseline.
print("Installing dependencies...")
pip_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REPO_DIR / "requirements.txt"),
        "python-docx",
        "gradio==6.16.0",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
if pip_result.returncode != 0:
    print(pip_result.stdout)
    raise RuntimeError("Dependency installation failed. The complete pip output is shown above.")
print("✓ Dependencies installed (Gradio 6.16.0)")

# 6) Launch the temporary authenticated Gradio UI.
env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["TRANSCRIBER_DRIVE_ROOT"] = "/content/drive/MyDrive"
env["TRANSCRIBER_UI_PASSWORD"] = secrets.token_urlsafe(10)

print("\nStarting Interview Transcriber...")
subprocess.run([sys.executable, str(REPO_DIR / "ui.py")], env=env, check=True)